# Problem 007:  10 001st Prime

> By listing the first six prime numbers: $2, 3, 5, 7, 11$, and $13$, we can see that the $6$th prime is $13$.
>
> What is the $10001$st prime number?


## Naive approach $\mathcal O (N^{3/2}\sqrt{\ln N})$

It is not a trivial matter to determine an upper limit of values to search for such a prime.  A really trivial code would just be to recycle the prime generator and just put in maximum search values until you get a list of at least $N$ primes.  But there is something unsatisfying about that, even if it is how I did it the first time I was confronted with this problem...

Instead, let's write a prime generator that does not rely on using a maximum value.  First, the code here just loops over an incrementer and checks if every value is a prime or not.  If it is, the count increases and it is appended to a list of primes.  The `is_prime` function just checks if any previous prime, up to $\sqrt x$, divides $x$.

A rough estimate of the scaling is as follows.  To check the primality of $n$, there are $\sim \frac{\sqrt n}{\ln n}$ checks, i.e. - the number of primes below $\sqrt n$.  Iterating $n$ up to $M$ asymptotically goes as $\frac{M^{3/2}}{\ln M}$.  But how large is $M$?  Knowing that the number of primes below $x$ goes as $\frac{x}{\ln x}$, the inverse of this gives the number of the $n^\mbox{th}$ prime, which is $M \sim N \ln N$.  Plugging this value of $M$ yields the scaling of $N^{3/2}\sqrt{\ln N}$.

In [6]:
%%time

def is_prime(x: int, primes: list[int]) -> bool:
    i = 0
    p = primes[i]
    for p in primes:
        if x < p*p:
            break
        if x % p == 0:
            return False
    return True

def p007_try1(N: int) -> int:
    primes = [2]
    np = 1
    i = 1
    while np < N:
        i += 2
        if is_prime(i, primes):
            primes.append(i)
            np += 1

    return primes[-1]


N = 10_001
ans = p007_try1(N)
print(ans)

104743
CPU times: user 131 ms, sys: 1.02 ms, total: 132 ms
Wall time: 133 ms


## Faster approach $\mathcal O (N \ln N \ln\ln N)$

The scaling above works fine for finding the $10001$st prime, but can be improved.  A sieve can be implemented in a dynamic way by shifting and increasing the range by a factor of two at each step.  In this way, the sieve first checks from $3$--$4$, then $5$--$8$, then $9$--$16$, etc. The computational cost of this is not horribly more than running a single sieve of size $2^n$, where $n$ is the smallest number that contains the desired prime.  The added complication over the typical sieve is in the indexing of a sieve that does not start at 0.

A nice feature of this sieve is that none of the primes in the new range will sieve any values in the current range.  Thus, first all the previous primes up to $2^{n/2}$ are used to set the sieve, then any untouched values in the sieve are primes to collect.

As for time scaling, the time scaling of the sieve up to a value $M$ is $M\ln\ln M$, and, as discussed above, $M \sim N \ln N$.  Combining these two gives the scaling above.  The code below is noticeably faster when looking for the $1,000,001$st prime (1 s versus 1m).

In [19]:
%%time

def extend_primes(pow2: int, primes: list[int]) -> None:
    s = [True] * pow2
    # seive previous primes
    for p in primes[1:]:
        if p * p > 4*pow2:
            break
        i0 = ((2 * pow2) // p + 1) * p
        if i0 % 2 == 0:
            i0 += p
        i0 = (i0 - 2*pow2) // 2
        l = (pow2 - i0-1) // p + 1
        s[i0::p] = [False] * l
    # search for new primes
    primes += [2*(pow2+i)+1 for i in range(pow2) if s[i]]
    return
    
def p007(N: int) -> int:
    pow2 = 1
    primes = [2]
    np = 1
    while True:
        extend_primes(pow2, primes)
        if len(primes) >= N:
            return primes[N-1]
        pow2 *= 2

N = 10_001
ans = p007(N)
print(ans)

104743
CPU times: user 6.32 ms, sys: 953 μs, total: 7.27 ms
Wall time: 7.21 ms
